# 02 · rent.com.au scraping, split between the group (Sept 2026 snapshot)

This notebook **plans, checks and merges** the scraping. The scraping itself runs in a terminal with
`scripts/scrape_rentcomau.py` (it takes a while and can be stopped and restarted at any time; it carries on
where it left off). Nothing in this notebook sends requests to rent.com.au.

**How the work is split.** The 656 suburbs from the course dataset are dealt out like cards (biggest first, in a
1-2-3-4-4-3-2-1 order), so everyone gets a fair mix of big and small suburbs and about the same amount of work.
The split is identical on every computer, so nobody scrapes the same suburb twice.

**Steps**

1. *Coordinator:* put everyone's short name in `TEAM` below and run section 1. Send each person their command.
2. *Everyone:* `git pull`, then check it works once (2 requests):
   `python3 scripts/scrape_rentcomau.py test --contact your.name@student.unimelb.edu.au`
3. *Everyone:* run your own command from section 1 (from the repo's top folder). About 20–30 minutes each.
4. *Everyone:* zip your folder `data/raw/rentcomau/2026-09/search/<your name>/` and send it to the coordinator
   **privately** (Drive/OneDrive/Teams). Never commit scraped data to GitHub (the repo is public).
5. *Coordinator:* unzip each folder into the same place, then run sections 3 and 4.
6. Re-run `02b_geo_visualisation`, `05b_accessibility_features` and `06_rent_drivers`: the 2026 listings appear
   in every map and table automatically.

**Optional extra stage.** `details` visits every listing's own page for bond, features (dishwasher, heating...),
furnished and the description: about 5 seconds per listing, so a few hours each. Everything else (rent, rooms,
type, coordinates, listing date, agency) already comes from the search pages.
`python3 scripts/scrape_rentcomau.py details --worker <your name> --contact <email>`

**Scraping politely (built into the script).** It reads robots.txt and skips anything disallowed; waits about
5 seconds between requests; identifies itself as a University of Melbourne student project with your email;
**stops immediately if the site refuses access**, and does not try to get around it; and never saves agents'
or landlords' names or phone numbers.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import pandas as pd
import config
import listings as L
import scrape_rentcomau as scr

SNAPSHOT = scr.DEFAULT_SNAPSHOT                      # "2026-09", the same for everyone
TEAM = ["kerri", "member2", "member3", "member4"]    # one short lowercase name per person

## 1. Who scrapes which suburbs

In [2]:
plan = scr.make_plan(scr.load_suburbs(), len(TEAM))
plan["worker"] = plan["shard"].map(dict(enumerate(TEAM, start=1)))
out = scr.snap_dir(SNAPSHOT)
out.mkdir(parents=True, exist_ok=True)
plan.to_csv(out / "plan.csv", index=False)

shares = plan.groupby(["shard", "worker"]).agg(suburbs=("suburb", "size"), est_pages=("est_pages", "sum"),
                                               domain_2025_listings=("listing_count", "sum"))
shares["est_minutes"] = (shares["est_pages"] * 5 / 60).round(0).astype(int)
shares

,,suburbs,est_pages,domain_2025_listings,est_minutes
shard,worker,,,,
1,kerri,163,241,3169,20
2,member2,163,243,3205,20
3,member3,164,245,3210,20
4,member4,164,240,3131,20


The page estimates use the 2025 Domain counts; rent.com.au often shows more listings per suburb
(Box Hill: 270 vs 111), so allow about 1.5× the minutes. Each person's command:

In [3]:
for shard, name in enumerate(TEAM, start=1):
    print(f"{name}:\n  python3 scripts/scrape_rentcomau.py search --shard {shard}/{len(TEAM)} "
          f"--worker {name} --contact YOUR_UNI_EMAIL\n")

kerri:
  python3 scripts/scrape_rentcomau.py search --shard 1/4 --worker kerri --contact YOUR_UNI_EMAIL

member2:
  python3 scripts/scrape_rentcomau.py search --shard 2/4 --worker member2 --contact YOUR_UNI_EMAIL

member3:
  python3 scripts/scrape_rentcomau.py search --shard 3/4 --worker member3 --contact YOUR_UNI_EMAIL

member4:
  python3 scripts/scrape_rentcomau.py search --shard 4/4 --worker member4 --contact YOUR_UNI_EMAIL



## 2. Check the parser (no internet)

Parses the two pages saved by the last `test` run (`data/raw/rentcomau/debug/`). Only summaries are shown here,
because the notebook is committed to a public repo.

In [4]:
try:
    rows, detail = scr.offline_check()
    parsed = pd.DataFrame(rows, columns=scr.SEARCH_COLUMNS)
    filled = parsed.notna().mean().mul(100).round(0).rename("% of listings with a value")
    display(filled[filled.index.isin(["weekly_rent", "bedrooms", "bathrooms", "carspaces", "property_type",
                                      "lat", "lon", "date_listed", "available_date", "agency",
                                      "walk_score", "transit_score"])].to_frame())
    if detail:
        print("Listing page fields found:", [k for k, v in detail.items() if v not in (None, "") and k != "listing_id"])
except SystemExit as e:
    print(e)

Saved search page: site says 270 listings; parsed 25 (method: page data)


,% of listings with a value
weekly_rent,100.0
bedrooms,100.0
bathrooms,100.0
carspaces,100.0
property_type,100.0
lat,100.0
lon,100.0
date_listed,100.0
available_date,100.0
agency,100.0


Listing page fields found: ['bond', 'weekly_rent_detail', 'furnished', 'partially_furnished', 'ensuites', 'structured_features', 'byline', 'description', 'photo_count', 'detail_scraped_date']


## 3. Progress

Counts the suburb files in `data/raw/rentcomau/2026-09/search/<name>/`. Each person can run
`python3 scripts/scrape_rentcomau.py status --workers 4` to see their own progress; here the coordinator sees
everyone's once the folders have been collected.

In [5]:
table, details = scr.status_table(SNAPSHOT, len(TEAM))
table.index = [f"{s} ({TEAM[s - 1]})" for s in table.index]
print("Listing-page details collected:", details or "none yet (optional stage)")
table

Listing-page details collected: none yet (optional stage)


,suburbs,done,listings_found,workers,percent_done
1 (kerri),163,0,0,-,0.0
2 (member2),163,0,0,-,0.0
3 (member3),164,0,0,-,0.0
4 (member4),164,0,0,-,0.0


## 4. Merge everyone's files and check them

Combines every person's files into `data/raw/rentcomau/2026-09/vic_rentals_rentcomau.csv`, the file that
`scripts/listings.py` reads. A listing can appear in more than one suburb's search (and so in two people's
files); one copy is kept, preferring the search of its own suburb.

In [6]:
merged = scr.merge(SNAPSHOT)
if merged is not None:
    domain_cols = pd.read_csv(config.DOMAIN_2025, nrows=0).columns
    print("\nDomain columns we also have:", sorted(set(domain_cols) & set(merged.columns)))
    print("Domain columns rent.com.au doesn't give:", sorted(set(domain_cols) - set(merged.columns)))
    display(merged.groupby("worker").agg(listings=("listing_id", "size"), suburbs=("search_suburb", "nunique"),
                                         median_weekly_rent=("weekly_rent", "median")))
    loaded = L.load_rentcomau_2026()
    print(f"scripts/listings.py loads {len(loaded):,} listings (inside their own suburb) as snapshot "
          f"{loaded['snapshot'].iloc[0]}")

Nothing to merge yet: no search files in data/raw/rentcomau/2026-09/search


## 5. Next

Re-run `02b_geo_visualisation` → `05b_accessibility_features` → `06_rent_drivers` (and `07_rent_forecast` once it is
finalised). Every map and table gains a 2026 version, and 02b draws the 2025→2026 rent change map. Only new
suburbs are sent to the routing service, because routes already calculated are saved.